In [1]:
PATH = '../../../Databases/world'

In [2]:
import pandas as pd
import numpy as np
import requests
import json
import sqlite3 as sql
from matplotlib import pyplot as plt
import seaborn as sns
PATH = '../../../Databases/world/'

In [4]:
def query(q, source='CO2_emission_1750-2024.sql'):
    with sql.connect(PATH+source) as conn:
        df = pd.read_sql(q, conn)
    return df

def execute(q, source='CO2_emission_1750-2024.sql'):
    with sql.connect(PATH+source) as conn:
        conn.execute(q)

    return 'done'

## Fetch the data for co2 emission
df = pd.read_csv("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.csv?v=1&csvType=full&useColumnShortNames=true", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

### Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/annual-co2-emissions-per-country.metadata.json?v=1&csvType=full&useColumnShortNames=true").json()

df.to_csv(PATH+'CO2_emission_1750-2024.csv')

with open(PATH+'CO2_emission_1750-2024_meta.json', 'w') as f:
    json.dump(metadata, f)

## Global temperature change to preindustrial average 1850-2026
df = pd.read_csv("https://ourworldindata.org/grapher/temperature-anomaly.csv?v=1&csvType=full&useColumnShortNames=false", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

### Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/temperature-anomaly.metadata.json?v=1&csvType=full&useColumnShortNames=false").json()

df.to_csv(PATH+'Global_temperature_change_1850-2026.csv')
with sql.connect(PATH+'Global_temperature_change_1850-2026.sql') as conn:
    df.to_sql('temp', conn, index=False, if_exists='replace')

with open(PATH+'Global_temperature_change_1850-2026_meta.json', 'w') as f:
    json.dump(metadata, f)

## Energy consumption 
dfen = pd.read_csv("https://ourworldindata.org/grapher/energy-mix.csv?v=1&csvType=full&useColumnShortNames=false&source=total&metric=total", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

### Fetch the metadata
metadataen = requests.get("https://ourworldindata.org/grapher/energy-mix.metadata.json?v=1&csvType=full&useColumnShortNames=false&source=total&metric=total").json()

#### Data_preparation

In [5]:
q = "SELECT * FROM CO2"
df = query(q)

In [6]:
df.shape,df.columns

((29384, 4),
 Index(['entity', 'code', 'year', 'emissions_total'], dtype='object'))

In [11]:
print('No of countries & regions: {}'.format(len(df.entity.unique())), '\n',
     'No of codes: {}'.format(len(df.code.unique())), '\n',
     'No of years: {}'.format(len(df.year.unique())))

No of countries & regions: 247 
 No of codes: 229 
 No of years: 275


with sql.connect(PATH+'CO2_emission_1750-2024.sql') as con: 
    df.to_sql('CO2',con=con, if_exists='replace')

In [12]:
agg = df.pivot_table(index='entity', values='emissions_total', aggfunc=['sum','count']).reset_index()

In [ ]:
agg.columns = agg.columns.droplevel(level=1)
agg.columns = ['entity', 'emissions_total', 'count']

In [103]:
agg['world_%'] = agg.emissions_total.map(lambda x: round((x / agg.loc[agg.entity == 'World', 'emissions_total'].values[0])*100,2))

In [105]:
agg_full = pd.merge(agg, df.drop_duplicates('entity'), on='entity', how='left', suffixes=(None,'_y'))
agg_full = agg_full.drop(['year', 'emissions_total_y'], axis=1)
agg_full[:3]

,entity,emissions_total,count,world_%,code
0,Afghanistan,2.426354e+08,76,0.01,AFG
1,Africa,5.412534e+10,141,2.93,OWID_AFR
2,Africa (GCP),5.412492e+10,175,2.93,OWID


In [107]:
agg_full.fillna({'code':'OWID'}, inplace=True)

In [108]:
groups = agg_full.loc[agg_full.code.str.contains('OWID')]

groups.to_csv(PATH+'CO2_emission_groups_1750-2024.csv')

In [110]:
countries = agg_full.loc[~agg_full.code.str.contains('OWID')]

countries.to_csv(PATH+'CO2_emission_countries_1750-2024.csv')

with sql.connect(PATH+'CO2_emission_1750-2024.sql') as conn:
    countries.to_sql('countries', conn, if_exists='replace', index=False)
    groups.to_sql('groups',conn, if_exists='replace', index=False)

In [78]:
sea = pd.read_csv(PATH+'sealevel/CSIRO_Recons_gmsl_yr_2015.csv')
sea['Time'] = sea.Time.map(lambda x: round(x, 0))
sea['Time'] = sea['Time'].astype(int)
sea.drop('GMSL uncertainty (mm)', axis=1, inplace=True)
sea.columns = ['year', 'height']

#with sql.connect(PATH+'CO2_emission_1750-2024.sql') as conn:
#    sea.to_sql('sea_level', con=conn, if_exists='replace', index=False)

In [ ]:
dfen.loc[dfen['Code'].isnull(), 'Code'] = 'OWID'
dfen.columns = ['entity', 'code', 'year', 'energy']
dfen.year = dfen.year.astype(int)
dfen.energy = dfen.energy.astype(float)
with sql.connect(PATH+'Global_energy_production_1800-2025.sql') as conn:
    dfen.to_sql('full_data', con=conn, index=False, if_exists='replace')

q = """CREATE VIEW owid AS 
SELECT * FROM full_data WHERE code LIKE '%OWID%'; """
execute(q, source='Global_energy_production_1800-2025.sql')
q = """CREATE VIEW countries AS 
SELECT * FROM full_data WHERE code NOT LIKE '%OWID%'; """
execute(q, source='Global_energy_production_1800-2025.sql')

with open(PATH+'Global_energy_production_1800-2025.json', 'w') as f:
    json.dump(metadataen, f)